# LoRA Fine-tuning Tutorial

This notebook demonstrates how to fine-tune ControlDiff using LoRA for parameter-efficient adaptation.

In [ ]:
import torch
import sys
sys.path.append('../src')

from models import UNet2DConditionModel
from training.lora import inject_lora_into_model, print_lora_summary, save_lora_weights, load_lora_weights

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Load Base Model

In [ ]:
# Load pretrained model
unet = UNet2DConditionModel().to(device)

# Optionally load pretrained weights
# unet.load_state_dict(torch.load('checkpoints/base_model.pt'))

print(f'Base model parameters: {sum(p.numel() for p in unet.parameters()):,}')

## 2. Inject LoRA

In [ ]:
# Inject LoRA into attention layers
unet = inject_lora_into_model(
    unet,
    rank=8,
    alpha=16.0,
    target_modules=['to_q', 'to_k', 'to_v', 'to_out.0'],
    dropout=0.0,
)

# Print summary
print_lora_summary(unet)

## 3. Setup Training

In [ ]:
# Get only LoRA parameters for training
lora_params = [p for p in unet.parameters() if p.requires_grad]

print(f'LoRA parameters: {sum(p.numel() for p in lora_params):,}')
print(f'Reduction: {100 * sum(p.numel() for p in lora_params) / sum(p.numel() for p in unet.parameters()):.2f}%')

# Create optimizer
optimizer = torch.optim.AdamW(lora_params, lr=5e-5)

## 4. Training Loop (Simplified)

In [ ]:
# Example training loop
unet.train()

# Dummy training data (replace with real data)
num_steps = 100

for step in range(num_steps):
    # Generate dummy batch
    latents = torch.randn(2, 4, 64, 64, device=device)
    timesteps = torch.randint(0, 1000, (2,), device=device)
    context = torch.randn(2, 77, 768, device=device)
    
    # Forward pass
    noise_pred = unet(latents, timesteps, context)
    
    # Compute loss (simplified)
    target_noise = torch.randn_like(noise_pred)
    loss = torch.nn.functional.mse_loss(noise_pred, target_noise)
    
    # Backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if step % 10 == 0:
        print(f'Step {step}: Loss = {loss.item():.4f}')

print('Training complete!')

## 5. Save LoRA Weights

In [ ]:
from pathlib import Path

output_dir = Path('../outputs/lora_weights')
output_dir.mkdir(parents=True, exist_ok=True)

save_lora_weights(unet, output_dir / 'lora_weights.pt')

print(f'LoRA weights saved to {output_dir}')

## 6. Load and Use LoRA Weights

In [ ]:
# Load base model again
new_unet = UNet2DConditionModel().to(device)

# Inject LoRA structure
new_unet = inject_lora_into_model(new_unet, rank=8, alpha=16.0)

# Load trained LoRA weights
load_lora_weights(new_unet, output_dir / 'lora_weights.pt')

print('LoRA weights loaded successfully!')

# Now use for inference
new_unet.eval()